# 01 — Data Preparation (shared by the whole team)
**Project:** Predicting Road Accident Severity Using Data Science — Milestone 1
**Owner:** Member 1 · **Data:** DfT Road Safety Open Data (STATS19), Great Britain, collisions 2024 and 2025

This notebook turns the two raw DfT collision files into **one prepared file** that every team member uses:
`collisions_2024_2025_prepared.csv.gz` (compressed CSV; `pd.read_csv` opens it directly).

What it does:
1. Loads and merges the 2024 and 2025 collision files.
2. Reads the official DfT data guide and uses it to decode every coded column.
3. Removes columns that are empty, outdated or just IDs.
4. Converts hidden missing codes (`-1`, `9`, `99` = unknown) to real missing values.
5. Creates the target variables (severity label, KSI, and severity-adjusted KSI).
6. Adds time features and grouped categories needed for the statistical tests.
7. Saves the prepared file, a missing-values report and a data dictionary.

**It does not impute or delete rows.** Those decisions belong to Member 3 (Data Cleaning), using the missing-values report produced here.

**How to run (Colab):** keep this notebook and the three DfT files in the shared Drive folder `Road_Accident_Project`, open the notebook in Colab and choose *Runtime → Run all*. Allow Drive access when asked. Outputs are saved back into the same Drive folder.

In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# Google Colab: connect Google Drive and use the shared project folder.
# On a laptop (Jupyter) the files are read from the notebook's own folder.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/Road_Accident_Project"   # change if your folder has another name
except ImportError:
    DATA_DIR = "."
FILES = {
    2024: "dft-road-casualty-statistics-collision-2024.csv",
    2025: "dft-road-casualty-statistics-collision-2025.csv",
}
GUIDE = "dft-road-casualty-statistics-road-safety-open-dataset-data-guide-2025.xlsx"

for f in list(FILES.values()) + [GUIDE]:
    status = "found" if os.path.exists(os.path.join(DATA_DIR, f)) else "MISSING - put it in DATA_DIR"
    print(f"{f}: {status}")

dft-road-casualty-statistics-collision-2024.csv: found
dft-road-casualty-statistics-collision-2025.csv: found
dft-road-casualty-statistics-road-safety-open-dataset-data-guide-2025.xlsx: found


## Step 1 — Load and merge the two years
ID columns are read as text so that codes such as `070971700` or `E06000001` are not damaged
(leading zeros removed, or `E` read as scientific notation).

In [2]:
TEXT_COLS = ["collision_index", "collision_ref_no", "local_authority_ons_district",
             "local_authority_highway", "local_authority_highway_current",
             "lsoa_of_accident_location", "date", "time"]

frames = []
for year, f in FILES.items():
    part = pd.read_csv(os.path.join(DATA_DIR, f), dtype={c: str for c in TEXT_COLS}, low_memory=False)
    print(f"{year}: {part.shape[0]:,} collisions, {part.shape[1]} columns")
    frames.append(part)

raw = pd.concat(frames, ignore_index=True)
print(f"\nMerged: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
print("Duplicate collision_index values:", raw["collision_index"].duplicated().sum())
print("Fully duplicated rows:", raw.duplicated().sum())

2024: 100,927 collisions, 44 columns


2025: 101,525 collisions, 44 columns

Merged: 202,452 rows x 44 columns
Duplicate collision_index values: 0


Fully duplicated rows: 0


In [3]:
raw.head()

,collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,number_of_casualties,date,day_of_week,time,local_authority_district,local_authority_ons_district,local_authority_highway,local_authority_highway_current,first_road_class,first_road_number,road_type,speed_limit,junction_detail_historic,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing_human_control_historic,pedestrian_crossing_physical_facilities_historic,pedestrian_crossing,light_conditions,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight
0,202417H100324,2024,17H100324,449865.0,528767.0,-1.22869,54.65155,17,2,1,5,12/01/2024,6,19:30,-1,E06000001,E06000001,E06000001,3,689,3,50,0,0,-1,0,-1,0,0,0,4,1,2,0,0,0,2,1,2,E01011963,-1,0,1.00000,0.00000
1,202417M110024,2024,17M110024,450069.0,519269.0,-1.22715,54.56618,17,3,2,1,01/07/2024,2,22:15,-1,E06000002,E06000002,E06000002,6,0,6,30,-1,0,-1,-1,-1,-1,-1,0,4,1,2,-1,-1,0,1,3,2,E01012026,3,1,0.00000,1.00000
2,2024480537342,2024,480537342,531643.0,180546.0,-0.10449,51.50852,48,3,1,2,18/09/2024,4,17:00,-1,E09000001,E09000001,E09000001,3,201,6,20,9,19,4,3,201,0,8,17,1,1,1,0,0,0,1,1,2,E01003934,-1,0,0.11617,0.88383
3,202417M105824,2024,17M105824,450326.0,519173.0,-1.22319,54.56529,17,3,2,2,04/05/2024,7,16:10,-1,E06000002,E06000002,E06000002,6,0,2,30,-1,13,4,6,0,-1,-1,0,1,1,1,-1,-1,0,1,3,2,E01012026,3,1,0.00000,1.00000
4,202417M114324,2024,17M114324,450741.0,520051.0,-1.21662,54.57314,17,3,1,1,30/08/2024,6,20:16,-1,E06000002,E06000002,E06000002,3,66,3,50,-1,13,2,6,0,-1,-1,0,4,1,1,-1,-1,0,2,1,2,E01035190,3,1,0.00000,1.00000


## Step 2 — Read the DfT data guide (the code dictionary)
The data files store categories as numbers. The guide's `2024_code_list` sheet gives the meaning of each code.
Only the rows for the **collision** table are needed.

In [4]:
guide = pd.read_excel(os.path.join(DATA_DIR, GUIDE), sheet_name="2024_code_list")
guide.columns = guide.columns.str.strip().str.lower()
guide = guide[guide["table"].str.strip().str.lower() == "collision"]

CODE_MAPS = {}
for field, rows in guide.groupby("field name"):
    rows = rows.dropna(subset=["code/format"])
    codes = pd.to_numeric(rows["code/format"], errors="coerce")
    mapping = {int(c): str(l).strip() for c, l in zip(codes, rows["label"]) if pd.notna(c) and pd.notna(l)}
    if mapping:
        CODE_MAPS[field.strip()] = mapping

print(f"Decoding maps found for {len(CODE_MAPS)} collision fields")
print("Example - light_conditions:", CODE_MAPS["light_conditions"])

Decoding maps found for 30 collision fields
Example - light_conditions: {1: 'Daylight', 4: 'Darkness - lights lit', 5: 'Darkness - lights unlit', 6: 'Darkness - no lighting', 7: 'Darkness - lighting unknown', -1: 'Data missing or out of range'}


## Step 3 — Remove columns that cannot be used

| Column(s) | Reason removed |
|---|---|
| `local_authority_district` | Discontinued: every value is `-1` |
| `*_historic` columns (4) | Old (pre-2024) coding; about 73% of rows are `-1` because most police forces now use the new coding |
| `special_conditions_at_site` | Old coding replaced by `carriageway_hazards`; about 73% `-1` |
| `collision_ref_no` | Police reference number; already contained in `collision_index` |
| `location_easting_osgr`, `location_northing_osgr` | Same location as `longitude`/`latitude` in another coordinate system |
| `first_road_number`, `second_road_number` | Road ID numbers, not road characteristics |
| `local_authority_highway`, `local_authority_highway_current` | Administrative codes duplicated by `local_authority_ons_district` |
| `enhanced_severity_collision` | Detailed injury code filled only by forces using injury-based reporting (27% blank); it repeats the severity, so it would also leak the answer |

In [5]:
DROP_COLS = ["local_authority_district",
             "junction_detail_historic", "pedestrian_crossing_human_control_historic",
             "pedestrian_crossing_physical_facilities_historic", "carriageway_hazards_historic",
             "special_conditions_at_site", "collision_ref_no",
             "location_easting_osgr", "location_northing_osgr",
             "first_road_number", "second_road_number",
             "local_authority_highway", "local_authority_highway_current", "enhanced_severity_collision"]

for c in ["local_authority_district", "special_conditions_at_site", "junction_detail_historic"]:
    print(f"{c}: {(raw[c] == -1).mean():.1%} of rows are -1")

df = raw.drop(columns=DROP_COLS).copy()
print(f"\nColumns: {raw.shape[1]} -> {df.shape[1]}")

local_authority_district: 100.0% of rows are -1
special_conditions_at_site: 72.7% of rows are -1
junction_detail_historic: 72.8% of rows are -1

Columns: 44 -> 30


## Step 4 — Fix 'not at a junction' recorded as missing
When a collision is **not at a junction** (`junction_detail = 0`), there is no junction control and no second road.
In those rows `junction_control` and `second_road_class` are often recorded as `-1` (missing), but the true value is
known: *Not at junction or within 20 metres* (code `0`). Recoding them prevents about 96,000 false missing values.

In [6]:
not_junction = df["junction_detail"] == 0
for col in ["junction_control", "second_road_class"]:
    fix = not_junction & (df[col] == -1)
    df.loc[fix, col] = 0
    print(f"{col}: {fix.sum():,} values recoded from -1 to 0 (not at junction)")

junction_control: 84,792 values recoded from -1 to 0 (not at junction)
second_road_class: 11,065 values recoded from -1 to 0 (not at junction)


## Step 5 — Convert hidden missing codes to real missing values
`pandas` does not know that `-1` means *missing* or that `9`/`99` means *unknown*, so `df.isnull()` would show almost nothing.
These codes are converted to `NaN` here. *Weather = 8 (Other)* and *Light = 7 (Darkness - lighting unknown)* are kept as real categories.

In [7]:
MISSING_CODES = {col: [-1] for col in df.select_dtypes("number").columns}
MISSING_CODES.update({
    "road_type": [-1, 9],                 # 9 = Unknown
    "junction_detail": [-1, 99],          # 99 = unknown (self reported)
    "junction_control": [-1, 9],          # 9 = unknown (self reported)
    "second_road_class": [-1, 9],         # 9 = Unknown (self rep only)
    "pedestrian_crossing": [-1, 99],
    "weather_conditions": [-1, 9],        # 9 = Unknown
    "road_surface_conditions": [-1, 9],
    "carriageway_hazards": [-1, 99],
    "urban_or_rural_area": [-1, 3],       # 3 = Unallocated
    "speed_limit": [-1, 99],
})
# columns where -1 is not a missing code
for col in ["collision_year", "number_of_vehicles", "number_of_casualties", "longitude", "latitude",
            "collision_adjusted_severity_serious", "collision_adjusted_severity_slight", "collision_injury_based"]:
    MISSING_CODES.pop(col, None)

before = df.isna().sum().sum()
for col, codes in MISSING_CODES.items():
    df[col] = df[col].where(~df[col].isin(codes), np.nan)
print(f"Missing values: {before:,} before -> {df.isna().sum().sum():,} after converting codes")

Missing values: 4 before -> 71,795 after converting codes


In [8]:
missing = (df.isna().sum().to_frame("missing_count")
             .assign(missing_pct=lambda t: (t["missing_count"] / len(df) * 100).round(2))
             .query("missing_count > 0")
             .sort_values("missing_count", ascending=False))
missing.to_csv(os.path.join(DATA_DIR, "missing_values_report.csv"))
missing

,missing_count,missing_pct
junction_detail,15135,7.48
trunk_road_flag,14188,7.01
pedestrian_crossing,10874,5.37
carriageway_hazards,9124,4.51
junction_control,7710,3.81
weather_conditions,5530,2.73
road_type,4521,2.23
road_surface_conditions,4350,2.15
second_road_class,321,0.16
light_conditions,22,0.01


## Step 6 — Decode codes into readable labels
For each coded column a new `<column>_label` column is added. The numeric column is **kept** because Milestone 2 models need numbers.

In [9]:
EXTRA_LABELS = {   # codes the guide lists without a label
    "carriageway_hazards": {0: "None"},
    "speed_limit": {20: "20 mph", 30: "30 mph", 40: "40 mph", 50: "50 mph", 60: "60 mph", 70: "70 mph"},
    "junction_control": {0: "Not at junction or within 20 metres"},
    "second_road_class": {0: "Not at junction or within 20 metres"},
}
DECODE = ["collision_severity", "day_of_week", "police_force", "first_road_class", "road_type", "speed_limit",
          "junction_detail", "junction_control", "second_road_class", "pedestrian_crossing",
          "light_conditions", "weather_conditions", "road_surface_conditions", "carriageway_hazards",
          "urban_or_rural_area", "did_police_officer_attend_scene_of_accident", "trunk_road_flag",
          "collision_injury_based"]

for col in DECODE:
    mapping = {**CODE_MAPS.get(col, {}), **EXTRA_LABELS.get(col, {})}
    df[col + "_label"] = df[col].map(mapping)
    unmapped = df.loc[df[col].notna() & df[col + "_label"].isna(), col].unique()
    if len(unmapped):
        print(f"WARNING {col}: codes without a label -> {unmapped}")
print("Decoding finished.")
df[["light_conditions", "light_conditions_label", "weather_conditions", "weather_conditions_label"]].head()

Decoding finished.


,light_conditions,light_conditions_label,weather_conditions,weather_conditions_label
0,4.0,Darkness - lights lit,1.0,Fine no high winds
1,4.0,Darkness - lights lit,1.0,Fine no high winds
2,1.0,Daylight,1.0,Fine no high winds
3,1.0,Daylight,1.0,Fine no high winds
4,4.0,Darkness - lights lit,1.0,Fine no high winds


## Step 7 — Target variables
* `severity` — Fatal / Serious / Slight (the original 3 classes).
* `KSI` — **1 = Killed or Seriously Injured** (Fatal + Serious), **0 = Slight**. Fatal collisions are only about 1.5% of records,
  so the binary KSI target is used for the main analysis and the Milestone 2 model (following DfT practice and recent UK studies).
* `KSI_adjusted` — the **probability** that the collision is KSI after DfT's severity adjustment.

**Why the adjusted version matters:** police forces are moving from officer-judged severity to *injury-based* reporting (CRASH),
which records more injuries as serious. In this data, injury-based reporting covers about 59% of 2024 collisions but about 87% of 2025
collisions. Raw KSI counts therefore rise partly because of the recording change. **Use `KSI_adjusted` when comparing 2024 with 2025.**

In [10]:
df["severity"] = pd.Categorical(df["collision_severity_label"], ["Fatal", "Serious", "Slight"], ordered=True)
df["KSI"] = df["collision_severity"].isin([1, 2]).astype(int)
df["KSI_adjusted"] = np.where(df["collision_severity"] == 1, 1.0, df["collision_adjusted_severity_serious"])

print(pd.crosstab(df["collision_year"], df["severity"], margins=True))
print()
print("Share of collisions recorded with injury-based reporting:")
print(df.groupby("collision_year")["collision_injury_based"].mean().map("{:.1%}".format))

severity        Fatal  Serious  Slight     All
collision_year                                
2024             1502    23567   75858  100927
2025             1453    25191   74881  101525
All              2955    48758  150739  202452

Share of collisions recorded with injury-based reporting:
collision_year
2024    58.8%
2025    86.5%
Name: collision_injury_based, dtype: str


In [11]:
year_summary = df.groupby("collision_year").agg(
    collisions=("KSI", "size"),
    KSI_raw=("KSI", "sum"),
    KSI_adjusted=("KSI_adjusted", "sum"),
)
year_summary["KSI_raw_%"] = (year_summary["KSI_raw"] / year_summary["collisions"] * 100).round(2)
year_summary["KSI_adjusted_%"] = (year_summary["KSI_adjusted"] / year_summary["collisions"] * 100).round(2)
year_summary["KSI_adjusted"] = year_summary["KSI_adjusted"].round(0).astype(int)
year_summary

,collisions,KSI_raw,KSI_adjusted,KSI_raw_%,KSI_adjusted_%
collision_year,,,,,
2024,100927,25069,26195,24.84,25.95
2025,101525,26644,27196,26.24,26.79


## Step 8 — Time features
* `hour`, `month`, `month_name`, `weekday` (Monday to Sunday), `is_weekend`, `season`
* `time_band`: Night (00–05), Morning (06–11), Afternoon (12–17), Evening (18–23)
* `rush_hour`: Monday–Friday, 07:00–09:59 or 16:00–18:59. The dataset has no traffic volumes, so rush hour is used as a proxy for heavier traffic.

In [12]:
df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"], format="%d/%m/%Y %H:%M")
df["hour"] = df["datetime"].dt.hour
df["month"] = df["datetime"].dt.month
df["month_name"] = pd.Categorical(df["datetime"].dt.strftime("%b"),
                                  ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"], ordered=True)
WEEKDAYS = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
df["weekday"] = pd.Categorical(df["day_of_week_label"], WEEKDAYS, ordered=True)
df["is_weekend"] = df["weekday"].isin(["Saturday", "Sunday"]).astype(int)
df["season"] = df["month"].map({12:"Winter",1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",
                                6:"Summer",7:"Summer",8:"Summer",9:"Autumn",10:"Autumn",11:"Autumn"})
df["time_band"] = pd.cut(df["hour"], [-1, 5, 11, 17, 23], labels=["Night","Morning","Afternoon","Evening"])
df["rush_hour"] = ((df["is_weekend"] == 0) & (df["hour"].between(7, 9) | df["hour"].between(16, 18))).astype(int)

# check: the DfT day_of_week code must agree with the date
mismatch = (df["datetime"].dt.day_name() != df["day_of_week_label"]).sum()
print("Rows where day_of_week disagrees with the date:", mismatch)
df[["datetime", "hour", "weekday", "is_weekend", "rush_hour", "time_band", "season"]].head()

Rows where day_of_week disagrees with the date: 0


,datetime,hour,weekday,is_weekend,rush_hour,time_band,season
0,2024-01-12 19:30:00,19,Friday,0,0,Evening,Winter
1,2024-07-01 22:15:00,22,Monday,0,0,Evening,Summer
2,2024-09-18 17:00:00,17,Wednesday,0,1,Afternoon,Autumn
3,2024-05-04 16:10:00,16,Saturday,1,0,Afternoon,Spring
4,2024-08-30 20:16:00,20,Friday,0,0,Evening,Summer


## Step 9 — Grouped categories for the statistical tests
Some categories are very small (e.g. *Snowing + high winds*: 44 collisions). Chi-square tests need expected counts of at least 5 per cell,
so these grouped versions are added. The detailed `_label` columns stay available.

In [13]:
df["light_group"] = df["light_conditions"].map({1: "Daylight", 4: "Darkness", 5: "Darkness", 6: "Darkness", 7: "Darkness"})
df["darkness_lighting"] = df["light_conditions"].map({1: "Daylight", 4: "Dark - lit", 5: "Dark - unlit/none",
                                                      6: "Dark - unlit/none", 7: "Dark - unknown"})
df["weather_group"] = df["weather_conditions"].map({1: "Fine", 4: "Fine", 2: "Rain", 5: "Rain", 3: "Snow", 6: "Snow",
                                                    7: "Fog or mist", 8: "Other"})
df["high_winds"] = df["weather_conditions"].map({1:0, 2:0, 3:0, 4:1, 5:1, 6:1, 7:0, 8:0})
df["surface_group"] = df["road_surface_conditions"].map({1: "Dry", 2: "Wet or damp", 3: "Snow or ice", 4: "Snow or ice",
                                                         5: "Other hazard", 6: "Other hazard", 7: "Other hazard"})
df["junction_group"] = (df["junction_detail"].eq(0).map({True: "Not at junction", False: "At junction"})
                         .where(df["junction_detail"].notna()))
df["road_class_group"] = df["first_road_class"].map({1: "Motorway", 2: "Motorway", 3: "A road", 4: "B road",
                                                     5: "C road", 6: "Unclassified"})
df["area"] = df["urban_or_rural_area_label"]

for col in ["light_group", "weather_group", "surface_group", "junction_group", "road_class_group", "area"]:
    print(df[col].value_counts(dropna=False).to_dict())

{'Daylight': 144364, 'Darkness': 58066, nan: 22}
{'Fine': 164656, 'Rain': 24563, 'Other': 6366, nan: 5530, 'Fog or mist': 816, 'Snow': 521}
{'Dry': 147511, 'Wet or damp': 47667, nan: 4350, 'Snow or ice': 2581, 'Other hazard': 343}
{'At junction': 95845, 'Not at junction': 91472, nan: 15135}
{'A road': 87969, 'Unclassified': 73367, 'B road': 25652, 'C road': 8948, 'Motorway': 6510, nan: 6}
{'Urban': 134452, 'Rural': 67993, nan: 7}


## Step 10 — Columns that must NOT be used as model features (Milestone 2)
These are known only **after** the collision or are derived from the severity itself. Using them would leak the answer into the model.

| Column | Why it leaks |
|---|---|
| `collision_severity`, `severity`, `KSI`, `KSI_adjusted` | They are the target |
| `collision_adjusted_severity_serious`, `collision_adjusted_severity_slight` | Calculated from the injuries |
| `number_of_casualties` | Counted after the collision; more casualties go with more severe collisions |
| `did_police_officer_attend_scene_of_accident` | Police are more likely to attend serious collisions |
| `collision_index`, `lsoa_of_accident_location`, `local_authority_ons_district` | Identifiers, not causes |

`collision_injury_based` is not a cause of severity but changes how it is recorded. Keep it as a **control variable** when comparing years.

In [14]:
TARGET_COLS = ["collision_severity", "severity", "KSI", "KSI_adjusted"]
LEAKAGE_COLS = ["collision_adjusted_severity_serious", "collision_adjusted_severity_slight",
                "number_of_casualties", "did_police_officer_attend_scene_of_accident"]
ID_COLS = ["collision_index", "lsoa_of_accident_location", "local_authority_ons_district"]
print("Do not use as features:", TARGET_COLS + LEAKAGE_COLS + ID_COLS)

Do not use as features: ['collision_severity', 'severity', 'KSI', 'KSI_adjusted', 'collision_adjusted_severity_serious', 'collision_adjusted_severity_slight', 'number_of_casualties', 'did_police_officer_attend_scene_of_accident', 'collision_index', 'lsoa_of_accident_location', 'local_authority_ons_district']


## Step 11 — Data dictionary for the analysis variables
Saved as `data_dictionary.csv` for the report's Dataset section.

In [15]:
DESCRIPTIONS = {
    "collision_year": "Year of the collision (2024 or 2025)",
    "datetime": "Date and time of the collision",
    "severity": "Collision severity: worst injury in the collision",
    "KSI": "Target: 1 = killed or seriously injured, 0 = slight",
    "KSI_adjusted": "Probability of KSI after DfT severity adjustment",
    "collision_injury_based": "1 = severity from injury-based reporting (CRASH), 0 = officer judgement",
    "number_of_vehicles": "Number of vehicles involved",
    "number_of_casualties": "Number of casualties (leakage - not a model feature)",
    "speed_limit": "Speed limit of the road (mph)",
    "road_type": "Road layout (single/dual carriageway, roundabout ...)",
    "road_class_group": "Class of the first road (Motorway, A, B, C, Unclassified)",
    "junction_group": "At a junction or not",
    "junction_detail_label": "Type of junction",
    "junction_control_label": "How the junction is controlled",
    "pedestrian_crossing_label": "Pedestrian crossing facility within 50 m",
    "light_conditions_label": "Lighting at the time of the collision",
    "light_group": "Daylight or darkness",
    "weather_conditions_label": "Weather at the time of the collision",
    "weather_group": "Weather grouped (Fine, Rain, Snow, Fog, Other)",
    "road_surface_conditions_label": "Road surface condition",
    "surface_group": "Surface grouped (Dry, Wet, Snow or ice, Other hazard)",
    "carriageway_hazards_label": "Object or hazard in the carriageway",
    "area": "Urban or rural area",
    "trunk_road_flag_label": "Trunk road (national network) or not",
    "police_force_label": "Police force that recorded the collision",
    "hour": "Hour of the day (0-23)",
    "weekday": "Day of the week",
    "is_weekend": "1 = Saturday or Sunday",
    "rush_hour": "1 = weekday 07:00-09:59 or 16:00-18:59",
    "time_band": "Night / Morning / Afternoon / Evening",
    "month_name": "Month",
    "season": "Season (Winter = Dec-Feb)",
    "longitude": "Longitude of the collision",
    "latitude": "Latitude of the collision",
}
rows = []
for col, desc in DESCRIPTIONS.items():
    s = df[col]
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > 12:
        values = f"{s.min():g} to {s.max():g}"
    elif pd.api.types.is_datetime64_any_dtype(s):
        values = f"{s.min():%d %b %Y} to {s.max():%d %b %Y}"
    else:
        values = f"{s.nunique()} categories"
    kind = "Numeric" if pd.api.types.is_numeric_dtype(s) and s.nunique() > 12 else (
           "Date/time" if pd.api.types.is_datetime64_any_dtype(s) else "Categorical")
    rows.append({"variable": col, "description": desc, "type": kind, "values": values,
                 "missing_%": round(s.isna().mean() * 100, 2)})
data_dictionary = pd.DataFrame(rows)
data_dictionary.to_csv(os.path.join(DATA_DIR, "data_dictionary.csv"), index=False)
data_dictionary

,variable,description,type,values,missing_%
0,collision_year,Year of the collision (2024 or 2025),Categorical,2 categories,0.00
1,datetime,Date and time of the collision,Date/time,01 Jan 2024 to 31 Dec 2025,0.00
2,severity,Collision severity: worst injury in the collision,Categorical,3 categories,0.00
3,KSI,"Target: 1 = killed or seriously injured, 0 = s...",Categorical,2 categories,0.00
4,KSI_adjusted,Probability of KSI after DfT severity adjustment,Numeric,0 to 1,0.00
5,collision_injury_based,1 = severity from injury-based reporting (CRAS...,Categorical,2 categories,0.00
6,number_of_vehicles,Number of vehicles involved,Numeric,1 to 26,0.00
7,number_of_casualties,Number of casualties (leakage - not a model fe...,Numeric,1 to 142,0.00
8,speed_limit,Speed limit of the road (mph),Categorical,6 categories,0.00
9,road_type,"Road layout (single/dual carriageway, roundabo...",Categorical,5 categories,2.23


## Step 12 — Save the prepared file

In [16]:
OUT = os.path.join(DATA_DIR, "collisions_2024_2025_prepared.csv.gz")   # compressed; pandas reads it directly
df.to_csv(OUT, index=False)
print(f"Saved {OUT}: {df.shape[0]:,} rows x {df.shape[1]} columns ({os.path.getsize(OUT)/1e6:.1f} MB)")

Saved ./collisions_2024_2025_prepared.csv.gz: 202,452 rows x 68 columns (12.9 MB)


## How team members load the prepared data
Copy this into the first cell of your own Colab notebook:

```python
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/Road_Accident_Project"

import pandas as pd
df = pd.read_csv(f"{DATA_DIR}/collisions_2024_2025_prepared.csv.gz", parse_dates=["datetime"], low_memory=False)

# restore category order for tables and charts
df["severity"] = pd.Categorical(df["severity"], ["Fatal", "Serious", "Slight"], ordered=True)
df["weekday"] = pd.Categorical(df["weekday"], ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"], ordered=True)
df["month_name"] = pd.Categorical(df["month_name"], ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"], ordered=True)
df["time_band"] = pd.Categorical(df["time_band"], ["Night","Morning","Afternoon","Evening"], ordered=True)
```

**Rules for everyone**
* Analyse with the readable columns (`*_label`, `*_group`, `area`, `weekday` ...) and the targets `KSI` / `severity`.
* Compare 2024 with 2025 using `KSI_adjusted` as well as `KSI`.
* Missing values are still in the file. Member 3 decides how to handle them; until then, `dropna` only the columns your test uses.